# 08 Portfolio Composition — ETF Look-Through

Shows the **aggregated composition** of your portfolio after expanding ETF holdings into their
underlying constituents. A security like ASML that you hold both directly and through an ETF
is merged into a single total weight.

**What this notebook produces:**
1. Security look-through table (direct + ETF weight + total, sorted by total)
2. Breakdown by sector and industry
3. Breakdown by country and geographic region
4. Breakdown by currency (from the raw holdings)
5. Breakdown by asset class (Equity / ETF)
6. Breakdown by market cap tier (Large / Mid / Small)
7. Breakdown by beta bucket (vs S&P 500, yfinance)
8. Breakdown by ETF structure (accumulating / distributing)
9. Breakdown by ETF domicile (Ireland / Luxembourg / etc.)
10. Per-ETF coverage summary with staleness warnings

**Key question answered:** What is my real exposure to each security, sector, and country
once I look through the ETFs I hold?

In [ ]:
import json
import os
import sys
from datetime import date as _date
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from portfolio_sim import (
    ChainedConstituentProvider,
    CsvConstituentProvider,
    YahooFinanceMetadataProvider,
    YahooTopHoldingsProvider,
    aggregate_portfolio_composition,
    breakdown_by_asset_class,
    breakdown_by_beta_bucket,
    breakdown_by_country,
    breakdown_by_currency,
    breakdown_by_etf_domicile,
    breakdown_by_etf_structure,
    breakdown_by_industry,
    breakdown_by_market_cap_tier,
    breakdown_by_region,
    breakdown_by_sector,
)

pd.options.display.float_format = "{:,.2f}".format
plt.rcParams["figure.figsize"] = (10, 5)

## Inputs

Set the paths below before running. All paths under `data/private/` are gitignored.

- `HOLDINGS_PATH` — Parquet or CSV file in HOLDINGS_COLUMNS schema (output of
  `scripts/normalize_portfolio_inputs.py` or `scripts/parse_db_pdf.py`).
- `TICKER_MAP_PATH` — JSON mapping ISIN → Yahoo Finance ticker
  (`data/private/ticker_map.json`).
- `ETF_URLS_PATH` — JSON mapping ETF ISIN → CSV download URL for constituent data
  (`data/private/etf_download_urls.json`). Leave empty string to skip CSV look-through.
- `ETF_OVERRIDES_PATH` — JSON mapping ETF ISIN → `"accumulating"` or `"distributing"`
  (`data/private/etf_structure_overrides.json`). Optional.
- `SNAPSHOT_DATE` — ISO date for the holdings snapshot (used for staleness checks).

In [ ]:
HOLDINGS_PATH = Path(os.environ.get("HOLDINGS_PATH", "data/private/holdings.parquet"))
TICKER_MAP_PATH = Path(os.environ.get("TICKER_MAP_PATH", "data/private/ticker_map.json"))
ETF_URLS_PATH = Path(os.environ.get("ETF_URLS_PATH", "data/private/etf_download_urls.json"))
ETF_OVERRIDES_PATH = Path(
    os.environ.get("ETF_OVERRIDES_PATH", "data/private/etf_structure_overrides.json")
)
SNAPSHOT_DATE = os.environ.get("SNAPSHOT_DATE", str(_date.today()))
CACHE_DIR = Path("data/private/etf_constituents_cache")
META_CACHE = Path("data/private/security_metadata.json")
COVERAGE_WARN_THRESHOLD = 0.80  # flag ETFs with constituent coverage below 80%

print(f"Holdings:      {HOLDINGS_PATH}")
print(f"Ticker map:    {TICKER_MAP_PATH}")
print(f"ETF URLs:      {ETF_URLS_PATH}")
print(f"Snapshot date: {SNAPSHOT_DATE}")

In [ ]:
# Load holdings
if HOLDINGS_PATH.suffix == ".parquet":
    holdings_df = pd.read_parquet(HOLDINGS_PATH)
else:
    holdings_df = pd.read_csv(HOLDINGS_PATH)

print(f"Loaded {len(holdings_df)} holdings rows")
print(f"Total portfolio value: EUR {holdings_df['market_value'].sum():,.2f}")
holdings_df.head()

In [ ]:
# Load lookup files
def _load_json(path: Path) -> dict:
    return json.loads(path.read_text()) if path.exists() else {}


ticker_map = _load_json(TICKER_MAP_PATH)
etf_urls = _load_json(ETF_URLS_PATH)
etf_overrides = _load_json(ETF_OVERRIDES_PATH)

print(f"Ticker map entries: {len(ticker_map)}")
print(f"ETF download URLs:  {len(etf_urls)}")
print(f"ETF structure overrides: {len(etf_overrides)}")

In [ ]:
# Build providers
csv_provider = CsvConstituentProvider(url_map=etf_urls, cache_dir=CACHE_DIR)
yahoo_fallback = YahooTopHoldingsProvider(
    isin_to_ticker=ticker_map,
    reverse_ticker_map={v: k for k, v in ticker_map.items()},
)
constituent_provider = ChainedConstituentProvider([csv_provider, yahoo_fallback])

metadata_provider = YahooFinanceMetadataProvider(
    isin_to_ticker=ticker_map,
    cache_path=META_CACHE,
    etf_structure_overrides=etf_overrides,
)

print("Providers ready.")

In [ ]:
# Run look-through aggregation
result = aggregate_portfolio_composition(
    holdings_df,
    constituent_provider,
    metadata_provider,
    snapshot_date=SNAPSHOT_DATE,
)

print(f"Aggregated {len(result.securities)} unique securities (including _UNRESOLVED_ if any)")
print(f"ETFs with constituent data: {len(result.etf_coverage)}")

## Coverage Warnings

The table below flags ETFs where constituent data coverage is below the threshold
or where the constituent data is more than 90 days older than the snapshot date.
Low-coverage ETFs have a meaningful 'Unresolved / other' residual that cannot be
attributed to any specific security, sector, or country.

In [ ]:
if result.etf_coverage.empty:
    print("No ETF constituent data loaded (no ETF URL map configured, or no ETFs in holdings).")
else:
    cov = result.etf_coverage.copy()
    cov["coverage_pct"] = (cov["coverage_pct"] * 100).round(1)
    cov["low_coverage"] = cov["coverage_pct"] < COVERAGE_WARN_THRESHOLD * 100

    warnings = cov[cov["low_coverage"] | cov["is_stale"]]
    if not warnings.empty:
        print("⚠  Coverage / staleness warnings:")
        display(warnings[["etf_isin", "coverage_pct", "as_of", "is_stale", "source"]])
    else:
        print("All ETF constituent data is within coverage and freshness thresholds.")

    print("\nFull ETF coverage summary:")
    display(cov[["etf_isin", "coverage_pct", "as_of", "is_stale", "source"]])

## 1 — Security Look-Through Table

Each row is one underlying security. `direct_weight_pct` is the weight from shares
held outright; `etf_weight_pct` is the weight derived from ETF constituents;
`total_weight_pct` is the sum. The `_UNRESOLVED_` row represents ETF weight that
could not be attributed to a specific security due to incomplete constituent data.

In [ ]:
display_cols = [
    "isin", "name", "direct_weight_pct", "etf_weight_pct", "total_weight_pct",
    "sector", "country", "market_cap_tier",
]
display(result.securities[display_cols].head(30))

## 2 — Sector & Industry Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sector_df = breakdown_by_sector(result.securities)
axes[0].barh(sector_df["dimension_value"], sector_df["weight_pct"])
axes[0].set_xlabel("Portfolio weight (%)")
axes[0].set_title("By sector")
axes[0].invert_yaxis()

industry_df = breakdown_by_industry(result.securities).head(15)
axes[1].barh(industry_df["dimension_value"], industry_df["weight_pct"])
axes[1].set_xlabel("Portfolio weight (%)")
axes[1].set_title("By industry (top 15)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("\nSector breakdown:")
display(sector_df)

## 3 — Country & Geographic Region Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

region_df = breakdown_by_region(result.securities)
axes[0].bar(region_df["dimension_value"], region_df["weight_pct"])
axes[0].set_ylabel("Portfolio weight (%)")
axes[0].set_title("By geographic region")
axes[0].tick_params(axis="x", rotation=30)

country_df = breakdown_by_country(result.securities).head(15)
axes[1].barh(country_df["dimension_value"], country_df["weight_pct"])
axes[1].set_xlabel("Portfolio weight (%)")
axes[1].set_title("By country (top 15)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("\nRegion breakdown:")
display(region_df)

## 4 — Currency Exposure

Based on the denomination currency of each holding in the broker report.
This reflects the currency of the instrument itself, not the underlying asset currency.

In [ ]:
currency_df = breakdown_by_currency(holdings_df)

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(
    currency_df["weight_pct"],
    labels=currency_df["dimension_value"],
    autopct="%1.1f%%",
    startangle=90,
)
ax.set_title("Currency exposure")
plt.show()

display(currency_df)

## 5 — Asset Class Breakdown

In [ ]:
ac_df = breakdown_by_asset_class(result.securities)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(ac_df["dimension_value"], ac_df["weight_pct"])
ax.set_ylabel("Portfolio weight (%)")
ax.set_title("By asset class")
plt.tight_layout()
plt.show()

display(ac_df)

## 6 — Market Cap Tier

In [ ]:
cap_df = breakdown_by_market_cap_tier(result.securities)

fig, ax = plt.subplots(figsize=(6, 4))
order = [t for t in ["Large", "Mid", "Small", "Unknown"] if t in cap_df["dimension_value"].values]
cap_ordered = cap_df.set_index("dimension_value").reindex(order).reset_index()
ax.bar(cap_ordered["dimension_value"], cap_ordered["weight_pct"])
ax.set_ylabel("Portfolio weight (%)")
ax.set_title("By market cap tier")
plt.tight_layout()
plt.show()

display(cap_df)

## 7 — Beta Distribution (vs S&P 500, yfinance)

Beta is measured against the S&P 500 as reported by Yahoo Finance. For European
or emerging-market holdings, this benchmark may understate sensitivity to local
market moves. Treat portfolio-weighted beta as an approximate risk indicator,
not a precise measure.

In [ ]:
beta_df = breakdown_by_beta_bucket(result.securities)

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(beta_df["dimension_value"], beta_df["weight_pct"])
ax.set_xlabel("Portfolio weight (%)")
ax.set_title("By beta bucket (vs S&P 500, yfinance)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Portfolio-weighted beta (excluding unknowns and unresolved)
known = result.securities[
    result.securities["beta"].notna()
    & (result.securities["isin"] != "_UNRESOLVED_")
].copy()
if not known.empty:
    total_known_weight = known["total_weight_pct"].sum()
    if total_known_weight > 0:
        portfolio_beta = (known["beta"] * known["total_weight_pct"]).sum() / total_known_weight
        print(f"\nPortfolio-weighted beta (vs S&P 500, yfinance): {portfolio_beta:.2f}")
        print(f"(covers {total_known_weight:.1f}% of portfolio by weight)")

display(beta_df)

## 8 — ETF Structure (Accumulating vs Distributing)

> **Tax note (informational only):** ETF structure and domicile are shown for
> informational purposes. Accumulating ETFs domiciled in Germany may trigger
> *Vorabpauschale* (annual deemed income tax) under the German Investment Tax
> Reform Act 2018; distributing ETFs pay dividends which are taxed as received.
> **No Vorabpauschale or any other tax liability is calculated in this notebook.**
> The exact tax treatment depends on your jurisdiction, holding period, the
> *Basiszins* for the relevant year, and the applicable *Teilfreistellung* rate.
> Consult a qualified tax professional for your specific situation.

In [ ]:
struct_df = breakdown_by_etf_structure(result.securities)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(struct_df["dimension_value"], struct_df["weight_pct"])
ax.set_ylabel("Portfolio weight (%)")
ax.set_title("By ETF structure")
plt.tight_layout()
plt.show()

display(struct_df)

## 9 — ETF Domicile

Derived from the two-letter ISIN country prefix (e.g. IE → Ireland, LU → Luxembourg).
ETF domicile affects the withholding tax treatment of dividends held inside the fund;
Irish-domiciled ETFs benefit from a more favourable US dividend withholding treaty rate
than Luxembourg-domiciled equivalents.

In [ ]:
domicile_df = breakdown_by_etf_domicile(result.securities)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(domicile_df["dimension_value"], domicile_df["weight_pct"])
ax.set_xlabel("Portfolio weight (%)")
ax.set_title("By ETF domicile (from ISIN prefix)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

display(domicile_df)